In [18]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
import pandas as pd
import csv

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# Create an empty DataFrame (not necessary since we are loading data immediately)
df = pd.read_csv('url_topic_small.csv')

print("Initial Data:")
print(df.head())

# Remove duplicates based on the 'URL' column
df = df.drop_duplicates(subset=['URL'])

print("\nData After Removing Duplicates:")
print(df.head())

# Handle NaN values in the 'Topic' column
# Option 1: Remove rows with NaN values in the 'Topic' column
df = df.dropna(subset=['Topic'])

# Option 2: Alternatively, you can fill NaN values with a placeholder (e.g., 'Unknown')
# df['Topic'] = df['Topic'].fillna('Unknown')

# Convert columns to lists
urls = df['URL'].tolist()
topics = df['Topic'].tolist()

# Limit the printed output to avoid overwhelming the notebook
print("\nURLs (First 10):")
print(urls[:10])
print("\nTopics (First 10):")
print(topics[:10])

Initial Data:
                                                 URL  \
0          https://www.change.org/t/highbridge-en-us   
1  https://www.change.org/t/assistant-principal-e...   
2              https://www.change.org/t/oikawa-en-us   
3  https://www.change.org/t/power-of-the-people-e...   
4  https://www.change.org/t/lauderdale-county-ala...   

                                               Topic  
0  highbridge | Highbridge Community - Take Actio...  
1  assistant principal | Assistant Principal - Ad...  
2  Oikawa | Oikawa - Take Action on Change.org | ...  
3  Power of the people | Power of the People - Ta...  
4                    Lauderdale County Alabama |  |   

Data After Removing Duplicates:
                                                 URL  \
0          https://www.change.org/t/highbridge-en-us   
1  https://www.change.org/t/assistant-principal-e...   
2              https://www.change.org/t/oikawa-en-us   
3  https://www.change.org/t/power-of-the-people-e...   
4  htt

In [3]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(topics)

In [4]:
cosine_sim = cosine_similarity(tfidf_matrix)
print(cosine_sim)

[[1.         0.03954208 0.07309814 ... 0.03125683 0.         0.        ]
 [0.03954208 1.         0.07519793 ... 0.04356281 0.         0.        ]
 [0.07309814 0.07519793 1.         ... 0.08423958 0.         0.        ]
 ...
 [0.03125683 0.04356281 0.08423958 ... 1.         0.         0.        ]
 [0.         0.         0.         ... 0.         1.         0.4945459 ]
 [0.         0.         0.         ... 0.         0.4945459  1.        ]]


In [5]:
def rank_urls(index, cosine_sim_matrix):
    similarity_scores = cosine_sim_matrix[index]

    sorted_indices = np.argsort(-similarity_scores)

    return sorted_indices[1:]

In [6]:

def print_top_related_to_csv(urls, cosine_sim, top_n=5, filename="related_urls.csv"):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Source URL", "Target URL", "Score"])

        for index in range(len(urls)):
            ranked_indices = rank_urls(index, cosine_sim)
            for i in ranked_indices[:top_n]:
                similarity_score = cosine_sim[index][i]
                writer.writerow([urls[index], urls[i], f"{similarity_score:.3f}"])

print_top_related_to_csv(urls, cosine_sim)